In [1]:
%run "MLSD.py"

In [28]:
# test kepler problem ground truth
xps = get_data(N=5, tgt=-5, steps=100)
mdl = MLSD(phase_dim=6, lie_dim=6, base_NN=NN, energy_func=energy_kep, obs_list=[S1, S2, S3, T1, T2, T3])
opt = torch.optim.Adam(mdl.parameters(), lr=10e-3)

In [30]:
# test kepler problem
xps = get_data(N=5, tgt=-5, steps=100)
mdl = MLSD(phase_dim=6, lie_dim=6, base_NN=NN, dims=[20,40,20])
opt = torch.optim.Adam(mdl.parameters(), lr=10e-3)

In [31]:
mdl.loss(xps)

tensor(0.0697, dtype=torch.float64, grad_fn=<DivBackward0>)

In [32]:
mdl.entropy(xps)

tensor(1.3838, dtype=torch.float64, grad_fn=<MeanBackward0>)

In [33]:
for i in range(200):
    xps = get_data(N=500, tgt=-3, steps=100)
    mdl_loss, entropy = mdl.loss(xps), mdl.entropy(xps)
    loss = mdl_loss - 0.0001* entropy
    opt.zero_grad()
    loss.backward()
    opt.step()
    print(f"Step {i+1}, Loss: {mdl_loss.item():.4f}, Entropy: {entropy.item():.4f}")

Step 1, Loss: 0.0731, Entropy: 1.3843
Step 2, Loss: 0.0143, Entropy: 1.4821
Step 3, Loss: 0.0231, Entropy: 1.5034
Step 4, Loss: 0.0226, Entropy: 1.5514
Step 5, Loss: 0.0114, Entropy: 1.4811
Step 6, Loss: 0.0055, Entropy: 1.4689
Step 7, Loss: 0.0062, Entropy: 1.4420
Step 8, Loss: 0.0078, Entropy: 1.4384
Step 9, Loss: 0.0068, Entropy: 1.4431
Step 10, Loss: 0.0048, Entropy: 1.4429
Step 11, Loss: 0.0025, Entropy: 1.4188
Step 12, Loss: 0.0013, Entropy: 1.4292
Step 13, Loss: 0.0012, Entropy: 1.4547
Step 14, Loss: 0.0017, Entropy: 1.4092
Step 15, Loss: 0.0022, Entropy: 1.3825
Step 16, Loss: 0.0024, Entropy: 1.3801
Step 17, Loss: 0.0023, Entropy: 1.3893
Step 18, Loss: 0.0014, Entropy: 1.4054
Step 19, Loss: 0.0007, Entropy: 1.4233
Step 20, Loss: 0.0005, Entropy: 1.4349
Step 21, Loss: 0.0004, Entropy: 1.4762
Step 22, Loss: 0.0006, Entropy: 1.5301
Step 23, Loss: 0.0008, Entropy: 1.5785
Step 24, Loss: 0.0009, Entropy: 1.5880
Step 25, Loss: 0.0008, Entropy: 1.5630
Step 26, Loss: 0.0006, Entropy: 1.

In [9]:
mdl.entropy(xps)

tensor(1.3411, dtype=torch.float64, grad_fn=<MeanBackward0>)

In [48]:
mdl.obs_vals(xps), mdl.f[0]

(tensor([[-0.0064, -0.0078,  0.0020, -0.0095, -0.0061,  0.0043],
         [-0.0064, -0.0082,  0.0033, -0.0086, -0.0083,  0.0048],
         [-0.0059, -0.0059,  0.0020, -0.0083, -0.0058,  0.0035],
         ...,
         [-0.0067, -0.0075,  0.0033, -0.0107, -0.0075,  0.0035],
         [-0.0059, -0.0073, -0.0003, -0.0092, -0.0042,  0.0019],
         [-0.0062, -0.0060,  0.0011, -0.0075, -0.0055,  0.0026]],
        dtype=torch.float64, grad_fn=<ViewBackward0>),
 tensor([[ 0.0000e+00,  5.5511e-17, -2.2204e-16,  0.0000e+00,  0.0000e+00,
          -1.1102e-16],
         [ 2.0817e-17,  0.0000e+00, -1.7467e+00, -3.3899e-01, -1.4357e+00,
          -2.3586e+00],
         [ 2.2204e-16,  1.7467e+00,  0.0000e+00, -6.6903e-01,  4.4322e-01,
           3.3730e+00],
         [ 0.0000e+00,  3.3899e-01,  6.6903e-01,  0.0000e+00, -2.3130e-01,
           9.1258e-02],
         [ 0.0000e+00,  1.4357e+00, -4.4322e-01,  2.3130e-01,  0.0000e+00,
           3.5007e+00],
         [ 0.0000e+00,  2.3586e+00, -3.3730e+

In [36]:
# optimize a linear transformation of structure constants
M = torch.nn.Parameter(torch.randn(6,6, dtype=mdl.f.dtype))
M_opt = torch.optim.Adam([M], lr=10e-4)

In [37]:
for _ in range(100000):
    f_trans = torch.einsum(mdl.f, [0,1,2], M, [0,3], M, [1, 4], torch.linalg.inv(M), [5, 2], [3,4,5]) # linear transformed from true f
    loss = torch.mean((f_trans - t1)**2)
    M_opt.zero_grad()
    loss.backward()
    M_opt.step()
loss


tensor(0.0278, dtype=torch.float64, grad_fn=<MeanBackward0>)

In [20]:
f_trans[0].round(decimals=4), t1[0].round(decimals=4)

(tensor([[ 0.0000, -0.0000,  0.0000,  0.0000, -0.0000,  0.0000],
         [-0.0301, -0.0053,  0.7013,  0.2221,  0.0306,  0.1150],
         [-0.0017, -0.6611,  0.0052, -0.3473,  0.2059,  0.1386],
         [ 0.0409, -0.1553,  0.2719, -0.0588, -0.2117,  0.0242],
         [ 0.0135, -0.0180, -0.1736,  0.1263,  0.0532,  0.0537],
         [ 0.0110, -0.0337, -0.0629, -0.0075, -0.0176,  0.0057]],
        dtype=torch.float64, grad_fn=<RoundBackward1>),
 tensor([[ 0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  1.,  0.,  0.,  0.],
         [ 0., -1.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.]], dtype=torch.float64))

In [38]:
mdl.f[0].type(torch.float64).round(decimals=4), t1[0].round(decimals=4)

(tensor([[ 0.0000, -0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000, -0.4713, -1.2758,  2.0861, -0.5693],
         [ 0.0000,  0.4713,  0.0000,  0.4512, -0.6337,  0.7107],
         [ 0.0000,  1.2758, -0.4512,  0.0000, -0.7234,  0.4165],
         [-0.0000, -2.0861,  0.6337,  0.7234,  0.0000,  0.3433],
         [ 0.0000,  0.5693, -0.7107, -0.4165, -0.3433,  0.0000]],
        dtype=torch.float64, grad_fn=<RoundBackward1>),
 tensor([[ 0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  1.,  0.,  0.,  0.],
         [ 0., -1.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.]], dtype=torch.float64))

In [25]:
mdl.f[0].type(torch.float64).round(decimals=4), t1[0].round(decimals=4)

(tensor([[ 0.0000e+00, -0.0000e+00, -0.0000e+00,  0.0000e+00, -0.0000e+00,
           0.0000e+00],
         [ 0.0000e+00,  0.0000e+00, -9.9270e-01, -8.0000e-04,  2.1000e-03,
           3.2000e-03],
         [ 0.0000e+00,  9.9270e-01,  0.0000e+00, -1.7000e-03, -1.0900e-02,
          -6.2000e-03],
         [ 0.0000e+00,  8.0000e-04,  1.7000e-03,  0.0000e+00, -1.0000e-04,
           7.0000e-04],
         [ 0.0000e+00, -2.1000e-03,  1.0900e-02,  1.0000e-04,  0.0000e+00,
           8.2000e-03],
         [-0.0000e+00, -3.2000e-03,  6.2000e-03, -7.0000e-04, -8.2000e-03,
           0.0000e+00]], dtype=torch.float64, grad_fn=<RoundBackward1>),
 tensor([[ 0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  1.,  0.,  0.,  0.],
         [ 0., -1.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.]], dtype=torch.float64))